# IT Service Ticket Classification — Preprocessing and Tokenization

## 1. Introduction

This notebook prepares the cleaned IT service ticket dataset for Hugging Face model training.

The workflow includes creating a stratified train/validation/test split, encoding categorical labels into numeric IDs, converting pandas DataFrames into Hugging Face Dataset objects, applying a DistilBERT tokenizer, and validating the resulting dataset structure.

The final output of this notebook is a tokenized dataset ready for transformer fine-tuning.


## 2. Import Libraries

In [1]:
# Import pandas for tabular data manipulation
import pandas as pd

# Import train_test_split to create stratified dataset splits
from sklearn.model_selection import train_test_split

# Import Dataset and DatasetDict to create Hugging Face-compatible datasets
from datasets import Dataset, DatasetDict

# Import AutoTokenizer to load the tokenizer associated with the selected transformer model
from transformers import AutoTokenizer

# Import json to save label mappings in a reusable format
import json

# Import os to create output folders if they do not exist
import os

## 3. Load Clean Dataset

In [2]:
# Define the path to the cleaned dataset
clean_data_path = "../data/processed/clean_tickets.csv"

# Load the cleaned dataset into a pandas DataFrame
df = pd.read_csv(clean_data_path)

# Display the first rows to validate the dataset structure
df.head()

,ticket_text,category
0,connection with icon icon dear please setup ic...,Hardware
1,work experience user work experience user hi w...,Access
2,requesting for meeting requesting meeting hi p...,Hardware
3,reset passwords for external accounts re expir...,Access
4,prod servers tunneling prod tunneling va la tu...,Hardware


In [3]:
# Display dataset dimensions to confirm the number of rows and columns
df.shape

(38607, 2)

In [4]:
# Display the class distribution before creating dataset splits
df["category"].value_counts()

category
Hardware                 13596
HR Support               10903
Access                    7111
Storage                   2777
Purchase                  2460
Administrative rights     1760
Name: count, dtype: int64

## 4. Define Label Encoding

In [5]:
# Define label mapping based on the actual categories available in the cleaned dataset
label2id = {
    "Hardware": 0,
    "HR Support": 1,
    "Access": 2,
    "Storage": 3,
    "Purchase": 4,
    "Administrative rights": 5
}

# Create reverse mapping to recover category names from numeric labels
id2label = {value: key for key, value in label2id.items()}

# Validate the label-to-ID mapping
label2id

{'Hardware': 0,
 'HR Support': 1,
 'Access': 2,
 'Storage': 3,
 'Purchase': 4,
 'Administrative rights': 5}

In [6]:
# Encode category names into numeric labels
df["label"] = df["category"].map(label2id)

# Confirm that all categories were mapped correctly
df["label"].isnull().sum()

np.int64(0)

In [7]:
# Display a sample of the encoded dataset
df.head()

,ticket_text,category,label
0,connection with icon icon dear please setup ic...,Hardware,0
1,work experience user work experience user hi w...,Access,2
2,requesting for meeting requesting meeting hi p...,Hardware,0
3,reset passwords for external accounts re expir...,Access,2
4,prod servers tunneling prod tunneling va la tu...,Hardware,0


## 5. Create Train / Validation / Test Split

Target split:

- Train: 70%
- Validation: 15%
- Test: 15%


In [8]:
# Create the training split with 70% of the dataset
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

# Split the remaining 30% into validation and test sets of 15% each
valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

In [9]:
# Display the number of records in each split
print("Train shape:", train_df.shape)
print("Validation shape:", valid_df.shape)
print("Test shape:", test_df.shape)

Train shape: (27024, 3)
Validation shape: (5791, 3)
Test shape: (5792, 3)


In [10]:
# Validate class distribution in the training set
train_df["category"].value_counts(normalize=True)

category
Hardware                 0.352168
HR Support               0.282416
Access                   0.184170
Storage                  0.071936
Purchase                 0.063721
Administrative rights    0.045589
Name: proportion, dtype: float64

In [11]:
# Validate class distribution in the validation set
valid_df["category"].value_counts(normalize=True)

category
Hardware                 0.352098
HR Support               0.282335
Access                   0.184251
Storage                  0.072008
Purchase                 0.063720
Administrative rights    0.045588
Name: proportion, dtype: float64

In [12]:
# Validate class distribution in the test set
test_df["category"].value_counts(normalize=True)

category
Hardware                 0.352210
HR Support               0.282459
Access                   0.184220
Storage                  0.071823
Purchase                 0.063709
Administrative rights    0.045580
Name: proportion, dtype: float64

## 6. Save Split Files

In [13]:
# Create the output folder for dataset splits if it does not exist
os.makedirs("../data/splits", exist_ok=True)

# Save the training split as a CSV file
train_df.to_csv("../data/splits/train.csv", index=False)

# Save the validation split as a CSV file
valid_df.to_csv("../data/splits/validation.csv", index=False)

# Save the test split as a CSV file
test_df.to_csv("../data/splits/test.csv", index=False)

In [14]:
# Create the output folder for metadata if it does not exist
os.makedirs("../data/metadata", exist_ok=True)

# Save the label-to-ID mapping as a JSON file
with open("../data/metadata/label2id.json", "w") as file:
    json.dump(label2id, file, indent=4)

# Save the ID-to-label mapping as a JSON file
with open("../data/metadata/id2label.json", "w") as file:
    json.dump(id2label, file, indent=4)

## 7. Convert pandas DataFrames to Hugging Face Datasets

In [15]:
# Convert the training DataFrame into a Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))

# Convert the validation DataFrame into a Hugging Face Dataset
valid_dataset = Dataset.from_pandas(valid_df.reset_index(drop=True))

# Convert the test DataFrame into a Hugging Face Dataset
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

# Combine all splits into a Hugging Face DatasetDict
dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": valid_dataset,
    "test": test_dataset
})

# Display the Hugging Face DatasetDict structure
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['ticket_text', 'category', 'label'],
        num_rows: 27024
    })
    validation: Dataset({
        features: ['ticket_text', 'category', 'label'],
        num_rows: 5791
    })
    test: Dataset({
        features: ['ticket_text', 'category', 'label'],
        num_rows: 5792
    })
})

## 8. Load DistilBERT Tokenizer

In [16]:
# Define the pretrained model checkpoint
model_checkpoint = "distilbert-base-uncased"

# Load the tokenizer associated with the selected model checkpoint
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [17]:
os.makedirs("../models/tokenizer", exist_ok=True)
tokenizer.save_pretrained("../models/tokenizer")

('../models/tokenizer\\tokenizer_config.json',
 '../models/tokenizer\\tokenizer.json')

## 9. Tokenize Dataset

In [18]:
# Define tokenization function for batches of ticket text
def tokenize_function(batch):
    # Tokenize ticket text using truncation and fixed-length padding
    return tokenizer(
        batch["ticket_text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

In [19]:
# Apply tokenization to all dataset splits
tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)

# Display the tokenized dataset structure
tokenized_datasets

Map:   0%|          | 0/27024 [00:00<?, ? examples/s]

Map:   0%|          | 0/5791 [00:00<?, ? examples/s]

Map:   0%|          | 0/5792 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['ticket_text', 'category', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 27024
    })
    validation: Dataset({
        features: ['ticket_text', 'category', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5791
    })
    test: Dataset({
        features: ['ticket_text', 'category', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5792
    })
})

## 10. Prepare Dataset Format for Model Training

In [20]:
# Remove text and category columns because the model only needs tokenized inputs and numeric labels
tokenized_datasets = tokenized_datasets.remove_columns(["ticket_text", "category"])

# Set PyTorch tensor format for model training
tokenized_datasets.set_format("torch")

# Display the final dataset structure
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 27024
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5791
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5792
    })
})

## 11. Validate Shapes and Examples

In [21]:
# Display one tokenized training example
tokenized_datasets["train"][0]

{'label': tensor(0),
 'input_ids': tensor([  101,  3229,  5227,  2005,  1037,  2862,  2030,  3075,  9432,  2255,
          7610,  3075, 17942,  3075,  6971,  3075,  6133,  4292,  3075,  3602,
          2609,  2609,  2609,  3640,  2430,  5527,  5792,  2592,  4784,  2609,
          6994,  5792,  6994,  4807,  3116,  6994,  3247,  2437,  2609,  7126,
          2967,  2147,  2780,  2591,  2967,  3745,  2592,  2147,  2362,  2742,
          2609,  2393, 13530,  8094,  2015, 20283,  6848,  4784,  3319, 10340,
          3745,  2592,  2562,  3543,  2060,   102,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,   

In [22]:
# Display the length of input_ids for one training example
len(tokenized_datasets["train"][0]["input_ids"])

256

In [23]:
# Display the length of attention_mask for one training example
len(tokenized_datasets["train"][0]["attention_mask"])

256

In [24]:
# Display the label for one training example
tokenized_datasets["train"][0]["label"]

tensor(0)

In [25]:
# Display final number of examples per split
print("Train examples:", len(tokenized_datasets["train"]))
print("Validation examples:", len(tokenized_datasets["validation"]))
print("Test examples:", len(tokenized_datasets["test"]))

Train examples: 27024
Validation examples: 5791
Test examples: 5792


In [26]:
# Create output folder for tokenized datasets
os.makedirs("../data/tokenized", exist_ok=True)

# Save tokenized Hugging Face dataset to disk
tokenized_datasets.save_to_disk("../data/tokenized/tickets_distilbert")

Saving the dataset (0/1 shards):   0%|          | 0/27024 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5791 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5792 [00:00<?, ? examples/s]

## 12.Conclusion

The prepared IT service ticket dataset was successfully transformed into a format suitable for transformer-based modeling using Hugging Face.

A stratified 70/15/15 split was applied to create training, validation, and test sets while preserving the original class distribution across all categories. This ensures consistent and reliable model evaluation.

Categorical labels were encoded into numeric representations and stored as reusable metadata, enabling seamless integration with downstream modeling and deployment stages.

The dataset was then converted into the Hugging Face DatasetDict format and tokenized using the distilbert-base-uncased tokenizer with a fixed maximum sequence length of 256 tokens. This step standardizes the input representation and ensures compatibility with transformer architectures.

Finally, the dataset was formatted into PyTorch tensors and persisted to disk, resulting in a fully reproducible and model-ready pipeline.

At this stage, the data is fully prepared for fine-tuning a DistilBERT model for multi-class IT service ticket classification.